# Comprehensive Data Cleaning Module - Food Intelligence System

**Detailed Tutorial and Examples**

This notebook provides a comprehensive guide to the data cleaning module with detailed examples, comparisons, and best practices for food ingredient text preprocessing.

**Author:** Food Intelligence System Team

---

## Table of Contents

1. [Setup and Installation](#setup)
2. [Module Overview](#overview)
3. [Individual Cleaning Techniques](#techniques)
4. [Cleaning Modes Comparison](#modes)
5. [Advanced Usage](#advanced)
6. [Performance Analysis](#performance)
7. [Real-World Examples](#examples)
8. [Best Practices](#practices)

## 1. Setup and Installation

First, let's import all required libraries and download NLTK data.

In [ ]:
# Import required libraries
import re
import string
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
import emoji
import contractions
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from textblob import TextBlob
from typing import List, Optional

print('All libraries imported successfully!')

In [ ]:
# Download required NLTK data
nltk_data = ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4']

for data in nltk_data:
    try:
        nltk.data.find(f'tokenizers/{data}' if 'punkt' in data else f'corpora/{data}')
        print(f'[OK] {data} already downloaded')
    except LookupError:
        print(f'[DOWNLOADING] {data}...')
        nltk.download(data, quiet=True)
        print(f'[OK] {data} downloaded')

print('\nAll NLTK data ready!')

## 2. Module Overview

The FoodTextCleaner class provides a comprehensive text cleaning pipeline.

In [ ]:
class FoodTextCleaner:
    """
    Comprehensive text cleaning pipeline for food ingredient data.
    """
    
    def __init__(self, lowercase=True, remove_html=True, remove_urls=True,
                 remove_punctuation=False, expand_contractions=True,
                 correct_spelling=False, remove_stopwords=False,
                 handle_emojis=True, tokenize=False, apply_stemming=False,
                 apply_lemmatization=False, preserve_food_terms=True):
        
        self.lowercase = lowercase
        self.remove_html = remove_html
        self.remove_urls = remove_urls
        self.remove_punctuation = remove_punctuation
        self.expand_contractions = expand_contractions
        self.correct_spelling = correct_spelling
        self.remove_stopwords = remove_stopwords
        self.handle_emojis = handle_emojis
        self.tokenize = tokenize
        self.apply_stemming = apply_stemming
        self.apply_lemmatization = apply_lemmatization
        self.preserve_food_terms = preserve_food_terms
        
        self.stop_words = set(stopwords.words('english'))
        self.stemmer = PorterStemmer()
        self.lemmatizer = WordNetLemmatizer()
        
        self.food_terms = {'oil', 'salt', 'pepper', 'sugar', 'butter', 'cream', 'milk',
                           'water', 'flour', 'egg', 'rice', 'meat', 'fish', 'chicken',
                           'beef', 'pork', 'cheese', 'bread', 'pasta', 'sauce', 'spice'}
        
        self.chat_word_dict = {'yum': 'yummy', 'delish': 'delicious', 'nom': 'eat',
                               'noms': 'food', 'veggies': 'vegetables', 'sammie': 'sandwich',
                               'sammy': 'sandwich', 'za': 'pizza', 'apps': 'appetizers',
                               'entree': 'main course'}
    
    def clean_text(self, text):
        if pd.isna(text) or not text:
            return ''
        text = str(text)
        
        if self.remove_html:
            text = self._remove_html_tags(text)
        if self.remove_urls:
            text = self._remove_urls(text)
        if self.handle_emojis:
            text = self._handle_emojis(text)
        if self.expand_contractions:
            text = self._expand_contractions(text)
        text = self._replace_chat_words(text)
        if self.lowercase:
            text = text.lower()
        text = self._remove_extra_whitespace(text)
        if self.remove_punctuation:
            text = self._remove_punctuation_text(text)
        if self.correct_spelling:
            text = self._correct_spelling(text)
        
        if self.tokenize or self.remove_stopwords or self.apply_stemming or self.apply_lemmatization:
            tokens = self._tokenize(text)
            if self.remove_stopwords:
                tokens = self._remove_stopwords_tokens(tokens)
            if self.apply_stemming:
                tokens = self._apply_stemming_tokens(tokens)
            if self.apply_lemmatization:
                tokens = self._apply_lemmatization_tokens(tokens)
            text = ' '.join(tokens)
        
        return text.strip()
    
    def clean_batch(self, texts):
        return [self.clean_text(text) for text in texts]
    
    def _remove_html_tags(self, text):
        soup = BeautifulSoup(text, 'html.parser')
        return soup.get_text(separator=' ')
    
    def _remove_urls(self, text):
        text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
        text = re.sub(r'www\.(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
        return text
    
    def _handle_emojis(self, text, mode='remove'):
        if mode == 'demojize':
            return emoji.demojize(text, delimiters=(' ', ' '))
        return emoji.replace_emoji(text, replace='')
    
    def _expand_contractions(self, text):
        try:
            return contractions.fix(text)
        except:
            return text
    
    def _replace_chat_words(self, text):
        words = text.split()
        return ' '.join([self.chat_word_dict.get(w.lower(), w) for w in words])
    
    def _remove_extra_whitespace(self, text):
        return re.sub(r'\s+', ' ', text).strip()
    
    def _remove_punctuation_text(self, text):
        return re.sub(r'[^\w\s/.\-]', '', text)
    
    def _correct_spelling(self, text):
        try:
            return str(TextBlob(text).correct())
        except:
            return text
    
    def _tokenize(self, text):
        return word_tokenize(text)
    
    def _remove_stopwords_tokens(self, tokens):
        if self.preserve_food_terms:
            return [t for t in tokens if t.lower() not in self.stop_words or t.lower() in self.food_terms]
        return [t for t in tokens if t.lower() not in self.stop_words]
    
    def _apply_stemming_tokens(self, tokens):
        return [self.stemmer.stem(t) for t in tokens]
    
    def _apply_lemmatization_tokens(self, tokens):
        return [self.lemmatizer.lemmatize(t) for t in tokens]

print('FoodTextCleaner class defined successfully!')